# SDXL Studio — ปรับแต่งสำหรับ Google Colab T4 ⚡

ตั้งค่าเริ่มต้นให้เหมาะกับ T4 (VRAM 16 GB): ใช้ SSD-1B เพื่อความเร็ว, สร้างครั้งละ 1 ภาพ และเริ่มที่ 768×768

## วิธีใช้
1. เลือก **Runtime → Change runtime type → T4 GPU**
2. รันเซลล์ติดตั้ง dependencies; ถ้ามีข้อความให้รีสตาร์ต runtime ให้รีสตาร์ต แล้วรันเซลล์ตั้งค่าและหน้าจออีกครั้ง
3. กด **Load model** รอให้ขึ้นสถานะพร้อมใช้งาน
4. ใส่ Prompt แล้วกด **Generate** — ใช้ขนาด 768 และ Batch 1 ก่อน หากต้องการภาพใหญ่ค่อยเพิ่มเป็น 832 หรือ 1024

### เคล็ดลับ T4
- SSD-1B เร็วกว่าโมเดล SDXL ขนาดเต็ม; เลือก Animagine/SDXL Base เมื่อต้องการโมเดลเหล่านั้น
- ลด Batch เป็น 1 และลดความละเอียดก่อน หากพบ CUDA out of memory
- VAE slicing/tiling เปิดไว้เพื่อช่วยลดการใช้ VRAM; การสร้างภาพทำทีละคำขอเพื่อไม่ให้แย่งหน่วยความจำ
- แชร์ลิงก์ Gradio เป็นสาธารณะชั่วคราว อย่าใส่ข้อมูลส่วนตัวใน Prompt

In [ ]:
# ใช้ PyTorch ที่มากับ Colab; ไม่ติดตั้งทับด้วยแพ็กเกจ CUDA/xformers ซึ่งอาจทำให้ runtime ไม่เข้ากัน
!pip -q install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import gc
import random
import torch
import gradio as gr
from diffusers import (
    StableDiffusionXLPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    UniPCMultistepScheduler,
    DDIMScheduler,
)

if not torch.cuda.is_available():
    raise RuntimeError("ไม่พบ GPU — เลือก Runtime → Change runtime type → T4 GPU แล้วรันใหม่")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} | VRAM: {gpu_memory_gb:.1f} GB | dtype: float16")

MODELS = {
    "SSD-1B (เร็ว, แนะนำสำหรับ T4)": "segmind/SSD-1B",
    "Animagine XL 4.0": "cagliostrolab/animagine-xl-4.0",
    "Stable Diffusion XL Base 1.0": "stabilityai/stable-diffusion-xl-base-1.0",
}
SCHEDULERS = {
    "UniPC (แนะนำ, ใช้ Steps น้อยลงได้)": UniPCMultistepScheduler,
    "Euler Ancestral": EulerAncestralDiscreteScheduler,
    "Euler": EulerDiscreteScheduler,
    "DDIM": DDIMScheduler,
}
pipe = None

def load_model(model_label, scheduler_label):
    global pipe
    if not torch.cuda.is_available():
        raise gr.Error("ไม่พบ GPU กรุณาเลือก T4 GPU")

    # ปล่อยโมเดลก่อนหน้าออกจาก VRAM ก่อนโหลดโมเดลใหม่
    if pipe is not None:
        del pipe
        pipe = None
    gc.collect()
    torch.cuda.empty_cache()

    try:
        model_id = MODELS[model_label]
        scheduler = SCHEDULERS[scheduler_label].from_pretrained(model_id, subfolder="scheduler")
        new_pipe = StableDiffusionXLPipeline.from_pretrained(
            model_id,
            scheduler=scheduler,
            torch_dtype=torch.float16,
            use_safetensors=True,
        ).to("cuda")

        # ลด VRAM ที่ใช้ระหว่าง decode ภาพขนาดใหญ่
        new_pipe.enable_vae_slicing()
        new_pipe.enable_vae_tiling()

        # ใช้ xFormers เฉพาะเมื่อมีอยู่และเปิดใช้งานได้; มิฉะนั้นให้ Diffusers/PyTorch ใช้ SDPA
        attention_mode = "PyTorch SDPA"
        try:
            new_pipe.enable_xformers_memory_efficient_attention()
            attention_mode = "xFormers"
        except Exception:
            pass
        new_pipe.set_progress_bar_config(disable=True)
        pipe = new_pipe
        del new_pipe
        torch.cuda.empty_cache()
        return f"พร้อมใช้: {model_label} | {scheduler_label} | {attention_mode} | {gpu_memory_gb:.1f} GB VRAM"
    except Exception as exc:
        pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        message = str(exc)
        if "out of memory" in message.lower():
            message += " — VRAM ไม่พอ ลองรีสตาร์ต runtime หรือใช้ SSD-1B"
        raise gr.Error(f"โหลดโมเดลไม่สำเร็จ: {message}")

@torch.inference_mode()
def generate_images(prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed):
    if pipe is None:
        raise gr.Error("กด Load model ก่อนสร้างภาพ")
    if not prompt or not prompt.strip():
        raise gr.Error("กรุณากรอก Prompt")

    seed = int(seed)
    if seed == -1:
        seed = random.randint(0, 2**31 - 1)
    elif seed < 0:
        raise gr.Error("Seed ต้องเป็น -1 (สุ่ม) หรือเลขตั้งแต่ 0 ขึ้นไป")

    try:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        result = pipe(
            prompt=prompt.strip(),
            negative_prompt=negative_prompt.strip() or None,
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg_scale),
            num_images_per_prompt=int(batch_size),
            generator=generator,
        )
        images = result.images
        del result
        return images, f"Seed: {seed}"
    except Exception as exc:
        gc.collect()
        torch.cuda.empty_cache()
        message = str(exc)
        if "out of memory" in message.lower():
            message += " — ลด Batch เป็น 1 และขนาดภาพเป็น 768×768 แล้วลองใหม่"
        raise gr.Error(f"สร้างภาพไม่สำเร็จ: {message}")

In [ ]:
# ค่าตั้งต้นเน้นความเร็วและประหยัด VRAM สำหรับ T4
sizes = [640, 768, 832, 896, 1024]

with gr.Blocks(theme=gr.themes.Soft(), title="SDXL Studio for T4") as demo:
    gr.Markdown("# SDXL Studio ⚡\nปรับค่าเริ่มต้นสำหรับ T4 — แนะนำ SSD-1B, 768×768, Batch 1")

    with gr.Accordion("1. โหลดโมเดล", open=True):
        with gr.Row():
            model_choice = gr.Dropdown(choices=list(MODELS), value="SSD-1B (เร็ว, แนะนำสำหรับ T4)", label="โมเดล")
            scheduler_choice = gr.Dropdown(choices=list(SCHEDULERS), value="UniPC (แนะนำ, ใช้ Steps น้อยลงได้)", label="Scheduler")
        load_button = gr.Button("Load model", variant="secondary")
        status = gr.Textbox(label="สถานะ GPU / โมเดล", value="ยังไม่ได้โหลดโมเดล", interactive=False)
        load_button.click(load_model, inputs=[model_choice, scheduler_choice], outputs=status)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 2. ตั้งค่าภาพ")
            prompt = gr.Textbox(label="Prompt", lines=4, placeholder="อธิบายภาพที่ต้องการ")
            negative_prompt = gr.Textbox(label="Negative prompt (ไม่บังคับ)", lines=2, placeholder="สิ่งที่ไม่ต้องการในภาพ")
            with gr.Row():
                width = gr.Dropdown(choices=sizes, value=768, label="กว้าง (px)")
                height = gr.Dropdown(choices=sizes, value=768, label="สูง (px)")
            with gr.Row():
                steps = gr.Slider(minimum=10, maximum=40, step=1, value=20, label="Steps (น้อย = เร็ว)")
                cfg_scale = gr.Slider(minimum=1, maximum=10, step=0.5, value=5, label="CFG Scale")
            with gr.Row():
                batch_size = gr.Slider(minimum=1, maximum=2, step=1, value=1, label="Batch (T4 แนะนำ 1)")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = สุ่ม)")
            generate_button = gr.Button("Generate", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### 3. ผลลัพธ์")
            gallery = gr.Gallery(label="ภาพที่สร้าง", columns=1, height="auto")
            seed_info = gr.Textbox(label="Seed ที่ใช้", interactive=False)

    generate_button.click(
        fn=generate_images,
        inputs=[prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed],
        outputs=[gallery, seed_info],
    )

# คิวทีละงานช่วยป้องกันการเรียกใช้ VRAM พร้อมกันจนเต็ม
demo.queue().launch(share=True, inline=True)